# 🚀 XScale AI Engine

**One-click setup:** Run all cells to start the AI video upscaling engine.

**First-time setup:** Add your Firebase credentials to Colab Secrets (🔑 icon in sidebar):
- Secret name: `FIREBASE_SERVICE_ACCOUNT`
- Value: paste the entire JSON from your Firebase service account key file

In [ ]:
# ═══ Cell 1: Install Dependencies + Clone Engine ═══
!pip install -q fastapi uvicorn pydantic firebase-admin google-api-python-client google-auth
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

# Clone engine code from GitHub
import os
if os.path.exists('/content/engine'):
    !cd /content/engine && git pull -q
    print('✅ Engine updated')
else:
    !git clone https://github.com/YOUR_USERNAME/XScale-engine.git /content/engine
    print('✅ Engine cloned')

In [ ]:
# ═══ Cell 2: Download AI Model Binaries ═══
import os
import zipfile
import subprocess
import glob

os.makedirs('/content/realesrgan-ncnn-vulkan', exist_ok=True)
os.makedirs('/content/realcugan-ncnn-vulkan', exist_ok=True)

# Download and extract Real-ESRGAN
url1 = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesrgan-ncnn-vulkan-20220424-ubuntu.zip'
subprocess.run(['wget', '-q', url1, '-O', 'realesrgan.zip'])
with zipfile.ZipFile('realesrgan.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/realesrgan-ncnn-vulkan')
os.chmod('/content/realesrgan-ncnn-vulkan/realesrgan-ncnn-vulkan', 0o777)
print('✅ Real-ESRGAN ready')

# Download and extract Real-CUGAN
url2 = 'https://github.com/nihui/realcugan-ncnn-vulkan/releases/download/20220728/realcugan-ncnn-vulkan-20220728-ubuntu.zip'
subprocess.run(['wget', '-q', url2, '-O', 'realcugan.zip'])
with zipfile.ZipFile('realcugan.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/realcugan-ncnn-vulkan')

binary = glob.glob('/content/realcugan-ncnn-vulkan/**/realcugan-ncnn-vulkan', recursive=True)
if binary:
    os.chmod(binary[0], 0o777)
    expected = '/content/realcugan-ncnn-vulkan/realcugan-ncnn-vulkan'
    if binary[0] != expected and not os.path.exists(expected):
        os.symlink(binary[0], expected)
    print(f'✅ Real-CUGAN ready: {binary[0]}')
else:
    print('❌ Real-CUGAN binary not found')

In [ ]:
# ═══ Cell 3: Configure Firebase ═══
import sys, os, json

# Add engine to Python path
sys.path.insert(0, '/content/engine')

# Load Firebase creds from Colab Secrets
from google.colab import userdata
creds_json = userdata.get('FIREBASE_SERVICE_ACCOUNT')
creds_data = json.loads(creds_json)
with open('/content/firebase-creds.json', 'w') as f:
    json.dump(creds_data, f)

os.environ['FIREBASE_CRED_PATH'] = '/content/firebase-creds.json'
os.environ['FIREBASE_DB_URL'] = 'https://xscale-1eda4-default-rtdb.asia-southeast1.firebasedatabase.app'

print('✅ Firebase configured')

In [ ]:
# ═══ Cell 4: Start Engine ═══
# Replace with your Firebase User ID from the app
USER_UID = "110779368462193186388"

import bridge
bridge.start_engine(user_uid=USER_UID, port=8000)